In [1]:
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch


In [2]:
def bytes_moved(shape: int, dtype: torch.dtype = torch.float32) -> float:
    return float(3 * shape * dtype.itemsize / 1e9)

In [3]:
with open("logs/meta_2026-09-23_10-02-13_94561134.json", "r") as f:
    meta = json.load(f)
mem_bandwidth = meta["measured_copy_bandwidth_gbps"]

In [4]:
times = []
with open("logs/times_2026-09-23_10-02-13_94561134.jsonl", "r") as f:
    times = [json.loads(l) for l in f]
    
df = pd.DataFrame(times)
df["p20_s"] = df.times.apply(lambda x: x[0] / 1e3)
df["med_s"] = df.times.apply(lambda x: x[1] / 1e3)
df["p80_s"] = df.times.apply(lambda x: x[2] / 1e3)
df["shape"] = df["shape"].apply(lambda x: str(x[0]))
df["kernel_name"] = df["kernel_name"].str.replace("vec_add_fwd", "vec_add_cuda_v1")
df["flops"] = df["shape"]
df["bytes_moved_gb"] = df["shape"].apply(lambda x: bytes_moved(shape=int(x)))
df["bandwidth_gbps"] = df["bytes_moved_gb"] / df["med_s"]
df["bandwidth_gbps_p20"] = df["bytes_moved_gb"] / df["p20_s"]
df["bandwidth_gbps_p80"] = df["bytes_moved_gb"] / df["p80_s"]
df["bandwidth_gbps_high"] = df["bandwidth_gbps_p80"] - df["bandwidth_gbps"]
df["bandwidth_gbps_low"] = df["bandwidth_gbps_p20"] - df["bandwidth_gbps"]
df.head()

,kernel_name,shape,dtype,seed,times,flops,bytes_moved,device,run_id,p20_s,med_s,p80_s,bytes_moved_gb,bandwidth_gbps,bandwidth_gbps_p20,bandwidth_gbps_p80,bandwidth_gbps_high,bandwidth_gbps_low
0,torch_add,1024,torch.float32,0,"[0.006111999973654747, 0.006175999995321035, 0...",1024,12288,cuda:0,94561134,0.000006,0.000006,0.000006,0.000012,1.989637,2.010471,1.979382,-0.010256,0.020834
1,torch_add,2048,torch.float32,0,"[0.004863999783992767, 0.00559999980032444, 0....",2048,24576,cuda:0,94561134,0.000005,0.000006,0.000006,0.000025,4.388572,5.052632,4.000000,-0.388572,0.664060
2,torch_add,4096,torch.float32,0,"[0.004921599850058555, 0.0054720002226531506, ...",4096,49152,cuda:0,94561134,0.000005,0.000005,0.000006,0.000049,8.982456,9.986996,8.293737,-0.688719,1.004541
3,torch_add,8192,torch.float32,0,"[0.0038591999094933272, 0.004031999967992306, ...",8192,98304,cuda:0,94561134,0.000004,0.000004,0.000004,0.000098,24.380953,25.472637,24.000000,-0.380953,1.091685
4,torch_add,16384,torch.float32,0,"[0.0041600000113248825, 0.004352000076323748, ...",16384,196608,cuda:0,94561134,0.000004,0.000004,0.000005,0.000197,45.176470,47.261538,41.234899,-3.941570,2.085069


In [5]:
df.kernel_name.unique()

<StringArray>
['torch_add', 'vec_add_cuda_v1', 'vec_add_triton_v1']
Length: 3, dtype: str

In [6]:
ADVERTISED_BW = 320

fig = go.Figure()


shapes = df["shape"].unique()

fig.add_traces(
    [
        go.Line(
            x=shapes, y=np.full_like(shapes, ADVERTISED_BW), name="Advertised Bandwidth"
        ),
        go.Line(
            x=shapes,
            y=np.full_like(shapes, mem_bandwidth),
            name="Measured Copy Bandwidth",
        ),
    ]
)
for kernel in df.kernel_name.unique():
    bw = df[df["kernel_name"] == kernel]["bandwidth_gbps"].values
    fig.add_trace(go.Line(x=shapes, y=bw, name=kernel))
fig.update_layout({"title":"VecAdd Bandwidth"})
fig.show()

/opt/venv/lib/python3.14/site-packages/plotly/graph_objs/_deprecations.py:378: DeprecationWarning: plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.

  warnings.warn(
/opt/venv/lib/python3.14/site-packages/plotly/graph_objs/_deprecations.py:378: DeprecationWarning: plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.

  warnings.warn(
/opt/venv/lib/python3.14/site-packages/plotly/graph_objs/_deprecations.py:378: DeprecationWarning: plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.

  warnings.warn(
/opt/venv/lib/python3.14/site-packages/plotly/graph_objs/_deprecations.py:378: DeprecationW

In [7]:
px.scatter(
    df,
    x="shape",
    y="bandwidth_gbps",
    error_y="bandwidth_gbps_high",
    error_y_minus="bandwidth_gbps_low",
    color="kernel_name",
    template="plotly_white",
    labels={"med": "Median runtime (s)"},
    title="VecAdd Sweep",
    facet_col="kernel_name",
).show()